In [ ]:
import os, re, sys, json, time, zipfile, secrets, string, socket, subprocess, textwrap

CONFIG = {
    "DOMAIN":        "example.com",
    "HOSTNAME":      "mail.example.com",
    "POSTMASTER":    "admin",
    "TLS_FLAVOR":    "notls",
    "ADMIN_PW":      "AdminPass123!",
    "WEBMAIL":       "none",
    "WEBDAV":        "none",
    "ANTIVIRUS":     "none",
    "FULL_TEXT_SEARCH": "off",
    "MESSAGE_SIZE_LIMIT": 50 * 1024 * 1024,
    "SUBNET":        "192.168.203.0/24",
    "RESOLVER_IP":   "192.168.203.254",
    "MAILU_VERSION": "2024.06",
    "PROJECT":       "mailu",
    "OUTDIR":        "mailu",
    "TEST_USERS":    {"alice": "AlicePass123!", "bob": "BobPass123!"},
    "TRY_DOCKER":    True,
}

BOLD, DIM, GRN, YEL, RED, CYN, RST = (
    "\033[1m", "\033[2m", "\033[92m", "\033[93m", "\033[91m", "\033[96m", "\033[0m")
def h(t):    print(f"\n{BOLD}{CYN}{'='*78}\n{t}\n{'='*78}{RST}")
def ok(t):   print(f"{GRN}  [ok]{RST} {t}")
def info(t): print(f"{DIM}  ...  {t}{RST}")
def warn(t): print(f"{YEL}  [!] {RST} {t}")
def err(t):  print(f"{RED}  [x] {RST} {t}")

def sh(cmd, timeout=900, check=False, env=None):
    """Run a shell command, stream nothing, return (rc, stdout+stderr)."""
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True,
                       timeout=timeout, env={**os.environ, **(env or {})})
    out = (p.stdout or "") + (p.stderr or "")
    if check and p.returncode != 0:
        raise RuntimeError(f"cmd failed ({p.returncode}): {cmd}\n{out}")
    return p.returncode, out

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

h("1. MAILU ARCHITECTURE  —  what you are about to deploy")
COMPONENTS = [
    ("front",    "nginx",    "Single entrypoint / reverse proxy. Terminates TLS and "
                             "proxies HTTP(80/443) + all mail ports (25/465/587/143/993/"
                             "110/995/4190). Does SMTP/IMAP auth handshakes against admin."),
    ("resolver", "unbound",  "Validating DNS resolver at a FIXED ip (192.168.203.254). "
                             "Every other container points its /etc/resolv.conf here so "
                             "DNSSEC/DANE/MTA-STS lookups work the same inside and out."),
    ("admin",    "admin",    "The brain: Flask web UI + REST API + the `flask mailu` CLI, "
                             "user/domain/alias database, DKIM key store, and the auth "
                             "endpoint the front asks 'is this login valid?'."),
    ("smtp",     "postfix",  "The MTA. Accepts inbound mail on 25, authenticated submission "
                             "on 587/465, hands mail to rspamd (milter) then to Dovecot via "
                             "LMTP for final delivery."),
    ("imap",     "dovecot",  "The MDA / mailbox store (Maildir). Serves IMAP/POP3, does the "
                             "SASL auth backend, ManageSieve (server-side filters), quotas."),
    ("antispam", "rspamd",   "Spam/DKIM/greylisting/DMARC/SPF engine. Signs outgoing mail "
                             "with DKIM, scores incoming, talks to redis for stats/state."),
    ("oletools", "oletools", "Sandbox that inspects Office attachments for malicious macros; "
                             "rspamd calls it during scanning."),
    ("redis",    "redis",    "Shared key/value store: rate limits, rspamd state, greylist db, "
                             "Roundcube cache, etc."),
]
for name, img, desc in COMPONENTS:
    print(f"  {BOLD}{name:<9}{RST}{DIM}({img}){RST}")
    for line in textwrap.wrap(desc, 70):
        print(f"      {line}")
info("Optional add-ons you can toggle in CONFIG: webmail (roundcube/snappymail), "
     "webdav (radicale), antivirus (clamav), fetchmail, full-text-search.")

In [ ]:
h("2. GENERATE  mailu.env")
os.makedirs(CONFIG["OUTDIR"], exist_ok=True)
SECRET_KEY = "".join(secrets.choice(string.ascii_letters + string.digits) for _ in range(16))

env_lines = f"""\
# --- Mailu {CONFIG['MAILU_VERSION']} environment (generated by the Colab tutorial) ---
# Generic
SECRET_KEY={SECRET_KEY}
SUBNET={CONFIG['SUBNET']}
DOMAIN={CONFIG['DOMAIN']}
HOSTNAMES={CONFIG['HOSTNAME']}
POSTMASTER={CONFIG['POSTMASTER']}
TLS_FLAVOR={CONFIG['TLS_FLAVOR']}
AUTH_RATELIMIT_IP=1000/minute
AUTH_RATELIMIT_USER=10000/day
DISABLE_STATISTICS=True
# Optional features
ADMIN=true
WEBMAIL={CONFIG['WEBMAIL']}
WEBDAV={CONFIG['WEBDAV']}
ANTIVIRUS={CONFIG['ANTIVIRUS']}
FULL_TEXT_SEARCH={CONFIG['FULL_TEXT_SEARCH']}
API=false
# Mail settings
MESSAGE_SIZE_LIMIT={CONFIG['MESSAGE_SIZE_LIMIT']}
RELAYNETS=
RELAYHOST=
FETCHMAIL_ENABLED=False
# Web settings
WEBROOT_REDIRECT=/webmail
WEB_ADMIN=/admin
WEB_WEBMAIL=/webmail
SITENAME=Mailu
WEBSITE=https://mailu.io
# Advanced
PASSWORD_SCHEME=PBKDF2
LOG_LEVEL=INFO
COMPOSE_PROJECT_NAME={CONFIG['PROJECT']}
"""
with open(f"{CONFIG['OUTDIR']}/mailu.env", "w") as f:
    f.write(env_lines)
ok(f"wrote {CONFIG['OUTDIR']}/mailu.env  (SECRET_KEY={SECRET_KEY})")

h("3. GENERATE  docker-compose.yml")
V = CONFIG["MAILU_VERSION"]
IMG = f"ghcr.io/mailu"

def svc(image, **extra):
    base = {"image": f"{IMG}/{image}:{V}", "restart": "always",
            "env_file": "mailu.env", "dns": [CONFIG["RESOLVER_IP"]],
            "logging": {"driver": "json-file",
                        "options": {"max-size": "10m", "max-file": "3"}}}
    base.update(extra)
    return base

compose = {
    "services": {
        "redis": {"image": "redis:alpine", "restart": "always",
                  "volumes": ["mailu_redis:/data"],
                  "dns": [CONFIG["RESOLVER_IP"]], "depends_on": ["resolver"]},
        "resolver": {"image": f"{IMG}/unbound:{V}", "env_file": "mailu.env",
                     "restart": "always",
                     "networks": {"default": {"ipv4_address": CONFIG["RESOLVER_IP"]}}},
        "front": svc("nginx",
                     depends_on=["resolver", "admin", "smtp", "imap", "antispam"],
                     ports=[
                         "80:80", "443:443",
                         "25:25", "465:465", "587:587",
                         "110:110", "995:995",
                         "143:143", "993:993",
                         "4190:4190",
                     ],
                     volumes=["mailu_certs:/certs", "mailu_overrides_nginx:/overrides:ro"]),
        "admin": svc("admin", depends_on=["redis", "resolver"],
                     volumes=["mailu_data:/data", "mailu_dkim:/dkim"]),
        "smtp": svc("postfix", depends_on=["front", "resolver"],
                    volumes=["mailu_mailqueue:/queue",
                             "mailu_overrides:/overrides:ro"]),
        "imap": svc("dovecot", depends_on=["front", "resolver"],
                    volumes=["mailu_mail:/mail",
                             "mailu_overrides:/overrides:ro"]),
        "antispam": svc("rspamd", depends_on=["front", "redis", "resolver", "oletools"],
                        volumes=["mailu_filter:/var/lib/rspamd",
                                 "mailu_overrides_rspamd:/overrides:ro"]),
        "oletools": {"image": f"{IMG}/oletools:{V}", "hostname": "oletools",
                     "restart": "always", "dns": [CONFIG["RESOLVER_IP"]],
                     "networks": {"default": None}},
    },
    "networks": {
        "default": {
            "driver": "bridge",
            "ipam": {"driver": "default",
                     "config": [{"subnet": CONFIG["SUBNET"]}]},
        }
    },
    "volumes": {v: None for v in [
        "mailu_redis", "mailu_certs", "mailu_overrides_nginx", "mailu_data",
        "mailu_dkim", "mailu_mailqueue", "mailu_overrides", "mailu_mail",
        "mailu_filter", "mailu_overrides_rspamd"]},
}

if CONFIG["WEBMAIL"] != "none":
    compose["services"]["webmail"] = svc(CONFIG["WEBMAIL"],
        depends_on=["front", "imap", "resolver"],
        volumes=["mailu_webmail:/data", "mailu_overrides_webmail:/overrides:ro"])
    compose["volumes"]["mailu_webmail"] = None
    compose["volumes"]["mailu_overrides_webmail"] = None

if CONFIG["ANTIVIRUS"] == "clamav":
    compose["services"]["antivirus"] = svc("clamav",
        depends_on=["resolver"], volumes=["mailu_filter_av:/data"])
    compose["volumes"]["mailu_filter_av"] = None

if CONFIG["WEBDAV"] == "radicale":
    compose["services"]["webdav"] = svc("radicale",
        depends_on=["resolver"], volumes=["mailu_dav:/data"])
    compose["volumes"]["mailu_dav"] = None

try:
    import yaml
except ImportError:
    sh("pip install -q pyyaml"); import yaml

raw = yaml.safe_dump(compose, sort_keys=False, default_flow_style=False, width=100)
raw = re.sub(r":\s*null\s*$", ":", raw, flags=re.M)
compose_path = f"{CONFIG['OUTDIR']}/docker-compose.yml"
with open(compose_path, "w") as f:
    f.write(raw)
ok(f"wrote {compose_path}  ({len(compose['services'])} services)")
info("services: " + ", ".join(compose["services"].keys()))

In [ ]:
h("4. VALIDATE the generated compose")
with open(compose_path) as f:
    parsed = yaml.safe_load(f)
assert "front" in parsed["services"], "front service missing!"
assert parsed["networks"]["default"]["ipam"]["config"][0]["subnet"] == CONFIG["SUBNET"]
assert parsed["services"]["resolver"]["networks"]["default"]["ipv4_address"] == CONFIG["RESOLVER_IP"]
ok("YAML parses; front/resolver/subnet wiring verified.")
print(DIM + "  ---- preview (first 30 lines) ----" + RST)
for line in raw.splitlines()[:30]:
    print("   " + line)
print(DIM + "  ---- (truncated) ----" + RST)

h("5. DNS RECORDS you must publish for a real deployment")
D, HN = CONFIG["DOMAIN"], CONFIG["HOSTNAME"]
dns = [
    ("A",   f"{HN}.",                 "<YOUR.SERVER.PUBLIC.IP>",
     "Points the mail host FQDN at your server."),
    ("MX",  f"{D}.",                  f"10 {HN}.",
     "Tells the world which host receives mail for the domain."),
    ("TXT", f"{D}.",                  f'"v=spf1 mx a:{HN} -all"',
     "SPF: only your MX/host may send as this domain."),
    ("TXT", f"_dmarc.{D}.",           f'"v=DMARC1; p=reject; rua=mailto:{CONFIG["POSTMASTER"]}@{D}; adkim=s; aspf=s"',
     "DMARC: reject spoofed mail, send reports to postmaster."),
    ("TXT", f"dkim._domainkey.{D}.",  '"v=DKIM1; k=rsa; p=<GENERATED-BY-MAILU>"',
     "DKIM: Mailu generates the key on first run — copy the exact record from "
     "the admin UI (Domains -> your domain -> 'Regenerate keys'/details) or "
     "`flask mailu config-export`."),
    ("CNAME", f"autoconfig.{D}.",     f"{HN}.",  "Thunderbird/Outlook autoconfig."),
    ("CNAME", f"autodiscover.{D}.",   f"{HN}.",  "Outlook autodiscover."),
    ("SRV", f"_submission._tcp.{D}.", f"0 1 587 {HN}.", "Client autoconfig: submission."),
    ("SRV", f"_imaps._tcp.{D}.",      f"0 1 993 {HN}.", "Client autoconfig: IMAPS."),
    ("TXT", f"_mta-sts.{D}.",         '"v=STSv1; id=<timestamp>"', "Enables MTA-STS policy."),
    ("CNAME", f"mta-sts.{D}.",        f"{HN}.", "Serves the MTA-STS policy file over HTTPS."),
]
print(f"  {'TYPE':<6}{'NAME':<28}VALUE")
print(f"  {'-'*4:<6}{'-'*4:<28}{'-'*5}")
for typ, name, val, why in dns:
    print(f"  {BOLD}{typ:<6}{RST}{name:<28}{val}")
    for line in textwrap.wrap(why, 62):
        print(f"        {DIM}{line}{RST}")
warn("Also set reverse DNS (PTR) for your server IP -> mail.<domain> at your host/ISP, "
     "or outbound mail will be flagged as spam.")

In [ ]:
h("6. LIVE RUN  —  best-effort (needs a Docker-capable runtime)")

def docker_capable():
    """Install docker if missing, start dockerd, and PROVE networking works."""
    if sh("docker version")[0] != 0:
        info("installing docker.io + compose v2 plugin ...")
        sh("apt-get update -qq && apt-get install -y -qq docker.io", timeout=600)
        sh("mkdir -p /usr/local/lib/docker/cli-plugins")
        sh("curl -fsSL "
           "https://github.com/docker/compose/releases/download/v2.29.7/"
           "docker-compose-linux-x86_64 "
           "-o /usr/local/lib/docker/cli-plugins/docker-compose "
           "&& chmod +x /usr/local/lib/docker/cli-plugins/docker-compose", timeout=300)
    if not os.path.exists("/var/run/docker.sock"):
        info("starting dockerd ...")
        subprocess.Popen("dockerd --host=unix:///var/run/docker.sock "
                         "--storage-driver=vfs > /var/log/dockerd.log 2>&1",
                         shell=True)
        for _ in range(30):
            if os.path.exists("/var/run/docker.sock"):
                break
            time.sleep(1)
    if sh("docker info")[0] != 0:
        return False, "dockerd did not come up (Colab usually blocks this)."
    sh("docker network rm _probe 2>/dev/null")
    if sh("docker network create _probe")[0] != 0:
        return False, "cannot create bridge networks (no NET_ADMIN in this sandbox)."
    sh("docker network rm _probe 2>/dev/null")
    rc, out = sh("docker run --rm hello-world", timeout=180)
    if rc != 0:
        return False, "cannot run containers: " + out.strip().splitlines()[-1:][0] if out else "unknown"
    return True, "docker networking works."

live_ok = False
if not CONFIG["TRY_DOCKER"]:
    info("TRY_DOCKER=False — skipping the live run.")
    capable, why = False, "disabled by config"
else:
    try:
        capable, why = docker_capable()
    except Exception as e:
        capable, why = False, f"probe error: {e}"

if not capable:
    warn(f"Docker not usable here: {why}")
    print(textwrap.dedent(f"""\
      {DIM}That's expected on stock Colab. Your generated deployment is ready to
      ship, though. On any Linux VPS with Docker + ports 25/465/587/993 open:{RST}

        scp -r {CONFIG['OUTDIR']}/ user@your-server:~/mailu && ssh user@your-server
        cd mailu
        docker compose -p {CONFIG['PROJECT']} up -d          # pull + start
        docker compose -p {CONFIG['PROJECT']} exec admin \\
            flask mailu admin {CONFIG['POSTMASTER']} {D} '{CONFIG['ADMIN_PW']}'
        # then open  http://your-server/admin  and publish the DNS records above.
    """))
else:
    ok(why)
    C = f"docker compose -p {CONFIG['PROJECT']} -f {compose_path} --env-file {CONFIG['OUTDIR']}/mailu.env"
    info("pulling images (this is a few hundred MB; give it a few minutes) ...")
    sh(f"{C} pull", timeout=1800)
    info("starting the stack ...")
    sh(f"{C} up -d", timeout=600)

    info("waiting for admin to become ready ...")
    ready = False
    for _ in range(40):
        if sh(f"{C} exec -T admin flask mailu --help")[0] == 0:
            ready = True; break
        time.sleep(5)
    if not ready:
        err("admin never became ready — dumping logs:")
        print(sh(f"{C} logs --tail 40 admin")[1])
    else:
        ok("admin is up. Provisioning domain + users via `flask mailu` ...")
        sh(f"{C} exec -T admin flask mailu domain {D}")
        sh(f"{C} exec -T admin flask mailu admin {CONFIG['POSTMASTER']} {D} '{CONFIG['ADMIN_PW']}'")
        for u, pw in CONFIG["TEST_USERS"].items():
            sh(f"{C} exec -T admin flask mailu user {u} {D} '{pw}'")
        ok(f"created domain {D}, admin {CONFIG['POSTMASTER']}@{D}, "
           f"users {', '.join(CONFIG['TEST_USERS'])}@{D}")

        import smtplib, imaplib, email
        from email.message import EmailMessage
        time.sleep(8)
        (alice, apw), (bob, bpw) = list(CONFIG["TEST_USERS"].items())[:2] or [("alice","x"),("bob","y")]
        alice_addr, bob_addr = f"{alice}@{D}", f"{bob}@{D}"
        token = secrets.token_hex(4)
        msg = EmailMessage()
        msg["From"], msg["To"] = alice_addr, bob_addr
        msg["Subject"] = f"Mailu Colab test {token}"
        msg.set_content(f"End-to-end delivery test, token={token}")

        sent = False
        try:
            info(f"submitting mail {alice_addr} -> {bob_addr} on 127.0.0.1:587 ...")
            s = smtplib.SMTP("127.0.0.1", 587, timeout=30); s.ehlo()
            if s.has_extn("starttls"):
                s.starttls(); s.ehlo()
            s.login(alice_addr, apw)
            s.send_message(msg); s.quit()
            sent = True; ok("mail submitted.")
        except Exception as e:
            err(f"submission failed: {e}")
            print(sh(f"{C} logs --tail 25 smtp")[1])

        if sent:
            info("polling bob's INBOX over IMAP (127.0.0.1:143) ...")
            found = False
            for _ in range(24):
                try:
                    m = imaplib.IMAP4("127.0.0.1", 143); m.login(bob_addr, bpw)
                    m.select("INBOX")
                    typ, data = m.search(None, f'(HEADER Subject "{token}")')
                    if typ == "OK" and data[0].split():
                        found = True; m.logout(); break
                    m.logout()
                except Exception:
                    pass
                time.sleep(5)
            if found:
                ok(f"DELIVERED ✅  bob received the message (token {token}). "
                   "Full SMTP->rspamd->LMTP->Dovecot->IMAP path works.")
            else:
                warn("mail was submitted but not seen in INBOX yet — check "
                     f"`{C} logs smtp` / `logs antispam` (greylisting/spam scoring).")

        try:
            if IN_COLAB:
                from google.colab.output import eval_js
                url = eval_js('google.colab.kernel.proxyPort(80)')
                ok(f"Admin UI (proxied):  {url}admin   "
                   f"login {CONFIG['POSTMASTER']}@{D} / {CONFIG['ADMIN_PW']}")
            else:
                ok(f"Admin UI:  http://127.0.0.1/admin   "
                   f"login {CONFIG['POSTMASTER']}@{D} / {CONFIG['ADMIN_PW']}")
        except Exception as e:
            info(f"couldn't create a proxy URL ({e}); use the mapped port 80 directly.")

In [1]:
h("7. DOWNLOAD your deployment bundle")
zip_path = "mailu_deployment.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(CONFIG["OUTDIR"]):
        for fn in files:
            fp = os.path.join(root, fn)
            z.write(fp, os.path.relpath(fp, "."))
ok(f"created {zip_path} containing docker-compose.yml + mailu.env")
try:
    if IN_COLAB:
        from google.colab import files
        files.download(zip_path)
        info("download started (check your browser).")
    else:
        info(f"not in Colab — grab the files under ./{CONFIG['OUTDIR']}/ and ./{zip_path}")
except Exception as e:
    info(f"auto-download unavailable ({e}); the files are on disk under ./{CONFIG['OUTDIR']}/")

h("DONE")
print(textwrap.dedent(f"""\
  You now have a faithful Mailu {V} deployment ({len(compose['services'])} services),
  a valid mailu.env, and the full DNS record set.

  Next steps for a REAL server:
    1. Point {HN} (A record) at a VPS you control (ports 25/465/587/993 open,
       and outbound 25 unblocked by the provider).
    2. scp the {CONFIG['OUTDIR']}/ folder over, set TLS_FLAVOR=letsencrypt in
       mailu.env, then `docker compose -p {CONFIG['PROJECT']} up -d`.
    3. Create the admin, log into /admin, copy the generated DKIM record, and
       publish all DNS records from section 5.
    4. Test your setup at https://www.mail-tester.com and https://mxtoolbox.com.
"""))


1. MAILU ARCHITECTURE  —  what you are about to deploy
  front    (nginx)
      Single entrypoint / reverse proxy. Terminates TLS and proxies
      HTTP(80/443) + all mail ports (25/465/587/143/993/110/995/4190). Does
      SMTP/IMAP auth handshakes against admin.
  resolver (unbound)
      Validating DNS resolver at a FIXED ip (192.168.203.254). Every other
      container points its /etc/resolv.conf here so DNSSEC/DANE/MTA-STS
      lookups work the same inside and out.
  admin    (admin)
      The brain: Flask web UI + REST API + the `flask mailu` CLI,
      user/domain/alias database, DKIM key store, and the auth endpoint the
      front asks 'is this login valid?'.
  smtp     (postfix)
      The MTA. Accepts inbound mail on 25, authenticated submission on
      587/465, hands mail to rspamd (milter) then to Dovecot via LMTP for
      final delivery.
  imap     (dovecot)
      The MDA / mailbox store (Maildir). Serves IMAP/POP3, does the SASL
      auth backend, ManageSieve (serve

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  ...  download started (check your browser).

DONE
You now have a faithful Mailu 2024.06 deployment (8 services),
a valid mailu.env, and the full DNS record set.

Next steps for a REAL server:
  1. Point mail.example.com (A record) at a VPS you control (ports 25/465/587/993 open,
     and outbound 25 unblocked by the provider).
  2. scp the mailu/ folder over, set TLS_FLAVOR=letsencrypt in
     mailu.env, then `docker compose -p mailu up -d`.
  3. Create the admin, log into /admin, copy the generated DKIM record, and
     publish all DNS records from section 5.
  4. Test your setup at https://www.mail-tester.com and https://mxtoolbox.com.

